# Notebook 02 — Split and Feature Preparation (leakage-free)

### What this notebook does
1. Keeps the **71 core questions** only (from `variable_dictionary.csv`, Notebook 01b)
2. Flags partial interviews (>50% of core questions blank) — same rule for stroke and non-stroke
3. Decodes coded answers (exercise frequency / duration, strength training) and fixes scale order
4. Fills **follow-up questions** logically (e.g. no drinks in past 30 days → 0 drinks per day) and checks every rule
5. Drops questions that only exist for an age group or are dates
6. Assigns every respondent to **train / validation / test (70/15/15)** once, and saves it
7. Saves the model-ready data and a feature configuration file for Notebook 03

### Why nothing here leaks information
Every step in this notebook uses **only the respondent's own answers** (e.g. "this person had no drinks, so drinks per day = 0").
Nothing is learned from other respondents — no averages, no models. Steps that *learn* from data
(imputation of remaining blanks, scaling, encoding) are done in Notebook 03, **inside** the training pipeline,
fitted on the training set only (reviewer comment #1).

**Input** — `data/processed/brfss_cleaned_v2.csv`, `outputs/variable_dictionary.csv`
**Output** — `data/processed/brfss_model_ready_v2.csv`, `data/processed/split_v2.csv`, `outputs/feature_config_v2.json`, `outputs/logical_fill_checks_v2.csv`

In [ ]:
import pandas as pd
import numpy as np
import os, json
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/stroke_prediction'
CLEANED_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_cleaned_v2.csv')
DICT_PATH    = os.path.join(BASE_DIR, 'outputs', 'variable_dictionary.csv')
READY_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
SPLIT_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'split_v2.csv')
CONFIG_PATH  = os.path.join(BASE_DIR, 'outputs', 'feature_config_v2.json')
CHECKS_PATH  = os.path.join(BASE_DIR, 'outputs', 'logical_fill_checks_v2.csv')

SEED              = 42      # fixed seed for the split (reproducibility)
CORE_MISSING_MAX  = 0.50    # partial-interview rule; the sensitivity analysis uses no filter

df = pd.read_csv(CLEANED_PATH)
df.insert(0, 'row_id', np.arange(len(df)))   # stable ID = row position in brfss_cleaned_v2.csv
var_dict = pd.read_csv(DICT_PATH)

CORE = var_dict.loc[var_dict['type'] == 'Core', 'feature'].tolist()
META = [c for c in df.columns if c.startswith('meta_')]
df = df[['row_id', 'stroke'] + CORE + META]

print(f"✓ Loaded {len(df):,} respondents")
print(f"✓ Core features: {len(CORE)} (module questions excluded from the primary analysis)")

### Step 1 — Flag partial interviews (same rule for both classes)

Respondents missing more than 50% of the core questions almost always stopped the phone interview early.
They are **flagged, not deleted**: the primary analysis excludes them, the sensitivity analysis keeps everyone.
The rule is measured **before** any filling, and is identical for stroke and non-stroke respondents (reviewer comment #2).

In [ ]:
df['meta_core_missing_pct'] = df[CORE].isna().mean(axis=1)
df['meta_keep_primary']     = df['meta_core_missing_pct'] <= CORE_MISSING_MAX

kept, dropped = df[df['meta_keep_primary']], df[~df['meta_keep_primary']]
print(f"Partial-interview rule: > {CORE_MISSING_MAX:.0%} of core questions blank")
print(f"  Flagged (excluded in primary analysis): {len(dropped):,} "
      f"(stroke={int(dropped['stroke'].sum()):,}, no stroke={int((dropped['stroke']==0).sum()):,})")
print(f"  Kept                                  : {len(kept):,}")
print(f"  Stroke prevalence — all: {df['stroke'].mean()*100:.2f}%   kept: {kept['stroke'].mean()*100:.2f}%")

### Step 2 — Decode coded answers and fix scale order

| Variable | Codebook coding | Becomes |
|---|---|---|
| EXEROFT1/2 | 1xx = times per week, 2xx = times per month | times per month |
| EXERHMM1/2 | H:MM (e.g. 130 = 1 h 30 min) | minutes |
| → combined | | `exercise_min_per_month` = times × minutes, both activities |
| STRENGTH | 1xx / 2xx as above, 888 = Never | `strength_times_per_month` (Never = 0) |
| MAXDRNKS | 88 = **Invalid response** (not "none") | missing |
| CHECKUP1 | 1–4 = years since check-up, **8 = Never** | Never moved to the end of the scale (5) |
| CHOLCHK3 | 2–6, 8 = years since check, **1 = Never** | 0 = <1 yr … 5 = 5+ yrs, 6 = Never |
| SEATBELT | 1–5 = Always…Never, **8 = never in a car** | missing (not on the seat-belt scale) |

In [ ]:
def times_per_month(s):
    """1xx = times per week (x 4.3), 2xx = times per month, anything else = NaN."""
    out = pd.Series(np.nan, index=s.index)
    wk = s.between(101, 199)
    mo = s.between(201, 299)
    out[wk] = (s[wk] - 100) * 4.3
    out[mo] = s[mo] - 200
    return out

def hmm_to_minutes(s):
    """H:MM coded as HMM (e.g. 130 = 1 h 30 min). Invalid minutes (>= 60) → NaN."""
    h, m = s // 100, s % 100
    valid = (s.between(1, 759) | s.between(800, 959)) & (m < 60)
    return (h * 60 + m).where(valid)

before = {c: int(df[c].notna().sum()) for c in ['EXEROFT1', 'EXERHMM1', 'EXEROFT2', 'EXERHMM2', 'STRENGTH']}

freq1, mins1 = times_per_month(df['EXEROFT1']), hmm_to_minutes(df['EXERHMM1'])
freq2, mins2 = times_per_month(df['EXEROFT2']), hmm_to_minutes(df['EXERHMM2'])

no_exercise   = df['EXERANY2'] == 0
no_second_act = (df['EXRACT22'] == 88) | no_exercise
freq1[no_exercise], mins1[no_exercise]   = 0, 0
freq2[no_second_act], mins2[no_second_act] = 0, 0

df['exercise_min_per_month'] = freq1 * mins1 + freq2 * mins2
df['EXRACT12'] = df['EXRACT12'].where(~no_exercise, 0)          # 0 = no exercise (own category)

df['strength_times_per_month'] = times_per_month(df['STRENGTH'])
df.loc[df['STRENGTH'] == 888, 'strength_times_per_month'] = 0   # 888 = Never

df['MAXDRNKS'] = df['MAXDRNKS'].replace({88.0: np.nan})         # 88 = invalid response
df['CHECKUP1'] = df['CHECKUP1'].replace({8.0: 5.0})             # Never → after "5+ years"
df['CHOLCHK3'] = df['CHOLCHK3'].map({2: 0, 3: 1, 4: 2, 5: 3, 6: 4, 8: 5, 1: 6})
df['SEATBELT'] = df['SEATBELT'].replace({8.0: np.nan})          # never drives / rides in a car

print("Non-missing before decoding:", before)
print(f"exercise_min_per_month   : non-missing {df['exercise_min_per_month'].notna().sum():,} | "
      f"median {df['exercise_min_per_month'].median():.0f} | zeros {(df['exercise_min_per_month']==0).sum():,}")
print(f"strength_times_per_month : non-missing {df['strength_times_per_month'].notna().sum():,} | "
      f"median {df['strength_times_per_month'].median():.0f}")
print("CHOLCHK3 after reorder   :", df['CHOLCHK3'].value_counts(dropna=False).sort_index().to_dict())
print("CHECKUP1 after reorder   :", df['CHECKUP1'].value_counts(dropna=False).sort_index().to_dict())

### Step 3 — Logical fill for follow-up questions (and a check for every rule)

A follow-up question is only asked when an earlier answer qualifies. When it was **not** asked, its true value is known:

| Follow-up | Asked only if… | If not asked, the value is |
|---|---|---|
| SMOKDAY2 (smoke now?) | smoked 100+ cigarettes | 3 = not at all |
| AVEDRNK3, DRNK3GE5, MAXDRNKS, DRNKDRI2 | drank in past 30 days | 0 |
| BPMEDS1 (BP medicine) | has hypertension | 0 = no |
| TOLDHI3, CHOLMED3 | cholesterol ever checked | 0 = no |
| ASTHNOW (still asthma) | ever had asthma | 0 = no |
| POORHLTH (days health stopped activities) | had ≥1 bad physical or mental day | 0 |
| PREGNANT | female aged 18–49 | 0 = no |
| COVIDSM1 (current symptoms) | ever had COVID | 0 = no |
| COVIDACT (activity limited) | has current symptoms | 3 = not at all |

The check table shows, for each rule, how many blanks it filled and how many genuine blanks remain.
`answered_but_skip` verifies the rule against the data: it counts people who answered although the rule says they
were skipped. Every rule here was checked on the full dataset (0 contradictions, except 9 noisy PREGNANT answers).
A rule "CHOLMED3 = 0 if never told cholesterol is high" was tested and **removed**: 19,902 such respondents did answer
CHOLMED3, so it is asked of everyone whose cholesterol was checked, and its blanks are genuine non-response.

In [ ]:
checks = []
def logical_fill(child, condition, value, rule):
    """Fill `child` with `value` where it is blank AND `condition` says it was not asked."""
    blank   = df[child].isna()
    to_fill = blank & condition.fillna(False)
    df.loc[to_fill, child] = value
    checks.append({
        'feature'           : child,
        'rule'              : rule,
        'blank_before'      : int(blank.sum()),
        'filled'            : int(to_fill.sum()),
        'blank_after'       : int(df[child].isna().sum()),
        'blank_after_pct'   : round(df[child].isna().mean() * 100, 2),
        # sanity check: answers that exist although the question should have been skipped
        'answered_but_skip' : int((~blank & condition.fillna(False) & (df[child] != value)).sum()),
    })

no_drinks = df['alcohol_days_per_month'] == 0
logical_fill('SMOKDAY2', df['SMOKE100'] == 0,               3, 'never smoked 100 cigarettes')
for c in ['AVEDRNK3', 'DRNK3GE5', 'MAXDRNKS', 'DRNKDRI2']:
    logical_fill(c, no_drinks,                              0, 'no drinks in past 30 days')
logical_fill('BPMEDS1',  df['hypertension'] == 0,           0, 'no hypertension')
logical_fill('TOLDHI3',  df['CHOLCHK3'] == 6,               0, 'cholesterol never checked')
logical_fill('CHOLMED3', df['CHOLCHK3'] == 6,               0, 'cholesterol never checked')
logical_fill('ASTHNOW',  df['ASTHMA3'] == 0,                0, 'never had asthma')
logical_fill('POORHLTH', (df['PHYSHLTH'] == 0) & (df['MENTHLTH'] == 0), 0, 'no bad health days')
logical_fill('PREGNANT', (df['gender'] == 1) | (df['age'] > 49), 0, 'male or over 49')
logical_fill('COVIDSM1', df['COVIDPO1'] == 0,               0, 'never had COVID')
logical_fill('COVIDACT', df['COVIDSM1'] == 0,               3, 'no current COVID symptoms')

checks_df = pd.DataFrame(checks)
checks_df.to_csv(CHECKS_PATH, index=False)
with pd.option_context('display.width', 200):
    print(checks_df.to_string(index=False))
print("\n'answered_but_skip' should be (close to) 0 — a large number would mean the skip rule is wrong.")

### Step 4 — Drop questions that cannot be used for everyone

| Dropped | Why |
|---|---|
| FALL12MN, FALLINJ5 | asked only of respondents aged 45+; falls are also a common **consequence** of stroke |
| SHINGLE2 | asked only of respondents aged 50+ |
| DIABAGE4 | age at diabetes diagnosis — meaningless for people without diabetes |
| FLSHTMY3, HIVTSTD3 | month/year dates of a vaccine or test |
| EXEROFT1/2, EXERHMM1/2, EXRACT22, STRENGTH | replaced by `exercise_min_per_month` and `strength_times_per_month` |

In [ ]:
DROP = {
    'FALL12MN': 'age-gated (45+); also a consequence of stroke',
    'FALLINJ5': 'follow-up of FALL12MN',
    'SHINGLE2': 'age-gated (50+)',
    'DIABAGE4': 'only defined for people with diabetes',
    'FLSHTMY3': 'date (month/year)',
    'HIVTSTD3': 'date (month/year)',
    'EXEROFT1': 'decoded into exercise_min_per_month',
    'EXERHMM1': 'decoded into exercise_min_per_month',
    'EXEROFT2': 'decoded into exercise_min_per_month',
    'EXERHMM2': 'decoded into exercise_min_per_month',
    'EXRACT22': 'second activity — folded into exercise_min_per_month',
    'STRENGTH': 'decoded into strength_times_per_month',
}
df = df.drop(columns=list(DROP))
print(f"✓ Dropped {len(DROP)} columns")

### Step 5 — Feature types and the two feature sets

**Types** decide how Notebook 03 encodes each feature: continuous and ordinal → numeric; binary → 0/1;
nominal (no natural order) → one-hot encoding (reviewer comment #7).

**Feature set A — all core features.**
**Feature set B — without likely consequences of stroke** (reviewer comment #3): treatments started after diagnosis
and functional limitations that stroke itself causes. If set B still performs well, the model is not just
recognising stroke survivors by their after-effects.

In [ ]:
CONTINUOUS = ['age', 'bmi', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'CHILDREN',
              'alcohol_days_per_month', 'AVEDRNK3', 'DRNK3GE5', 'MAXDRNKS', 'DRNKDRI2',
              'exercise_min_per_month', 'strength_times_per_month']
ORDINAL    = ['GENHLTH', 'CHECKUP1', 'CHOLCHK3', 'EDUCA', 'INCOME3', 'diabetes_status',
              'SMOKDAY2', 'USENOW3', 'COVIDACT', 'SEATBELT']
NOMINAL    = ['MARITAL', 'EMPLOY1', 'PRIMINS1', 'RENTHOM1', 'ECIGNOW2', 'PERSDOC3', 'EXRACT12']
BINARY     = ['gender', 'MEDCOST1', 'EXERANY2', 'hypertension', 'BPMEDS1', 'TOLDHI3', 'CHOLMED3',
              'CVDINFR4', 'CVDCRHD4', 'ASTHMA3', 'ASTHNOW', 'CHCSCNC1', 'CHCOCNC1', 'CHCCOPD3',
              'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4', 'VETERAN3', 'PREGNANT', 'DEAF', 'BLIND',
              'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'FLUSHOT7', 'PNEUVAC4',
              'HIVTST7', 'COVIDPO1', 'COVIDSM1']

POST_STROKE = {
    'CHOLMED3': 'cholesterol medication — routinely started after stroke',
    'BPMEDS1' : 'blood-pressure medication — routinely started after stroke',
    'DIFFWALK': 'difficulty walking — a common stroke consequence',
    'DIFFDRES': 'difficulty dressing/bathing — a common stroke consequence',
    'DIFFALON': 'difficulty doing errands alone — a common stroke consequence',
    'DECIDE'  : 'difficulty concentrating/remembering — a common stroke consequence',
    'EMPLOY1' : 'includes "unable to work" — often a result of stroke',
    'GENHLTH' : 'self-rated health — lowered by having had a stroke',
    'PHYSHLTH': 'days of poor physical health — affected by stroke',
    'POORHLTH': 'days health limited activities — affected by stroke',
}

FEATURES_A = CONTINUOUS + ORDINAL + NOMINAL + BINARY
FEATURES_B = [f for f in FEATURES_A if f not in POST_STROKE]

# Every remaining column must have exactly one type — no feature silently left out
present = [c for c in df.columns if c not in ('row_id', 'stroke') and not c.startswith('meta_')]
missing_type = sorted(set(present) - set(FEATURES_A))
extra_type   = sorted(set(FEATURES_A) - set(present))
dupes        = sorted({f for f in FEATURES_A if FEATURES_A.count(f) > 1})
assert not missing_type, f"Columns without a type: {missing_type}"
assert not extra_type,   f"Typed columns not in data: {extra_type}"
assert not dupes,        f"Columns with two types: {dupes}"

# Binary columns must really be 0/1
bad_binary = [c for c in BINARY if not set(df[c].dropna().unique()) <= {0, 1}]
assert not bad_binary, f"Binary columns with other values: {bad_binary}"

print(f"✓ All {len(FEATURES_A)} features typed: continuous {len(CONTINUOUS)}, ordinal {len(ORDINAL)}, "
      f"nominal {len(NOMINAL)}, binary {len(BINARY)}")
print(f"✓ Feature set A (all core)          : {len(FEATURES_A)}")
print(f"✓ Feature set B (no post-stroke)    : {len(FEATURES_B)}  (removed: {list(POST_STROKE)})")

### Step 6 — Remaining blanks (genuine non-response)

These are filled in Notebook 03 **inside** the training pipeline (fitted on the training set only), with a
"was missing" indicator so the model can tell a filled value from a real one.

In [ ]:
remaining = (df.loc[df['meta_keep_primary'], FEATURES_A].isna().mean() * 100).round(2)
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(f"Features with any blanks left (primary sample): {len(remaining)} of {len(FEATURES_A)}")
print(remaining.head(20).to_string())
print(f"\nOverall blank cells left: {df.loc[df['meta_keep_primary'], FEATURES_A].isna().mean().mean()*100:.2f}%")

### Step 7 — One train / validation / test split for everything

The split is made **once, on all respondents**, stratified by stroke, with a fixed seed. Every later notebook
(and both the primary and sensitivity analyses) uses this same assignment, so test respondents are never
used for training, tuning, imputation or feature choices (reviewer comments #1 and #5).

| Part | Share | Used for |
|---|---|---|
| train | 70% | fitting models, cross-validated tuning, imputation and encoding |
| validation | 15% | probability calibration and choosing the decision threshold |
| test | 15% | final results only — touched once |

In [ ]:
train_ids, temp_ids = train_test_split(df['row_id'], test_size=0.30,
                                       stratify=df['stroke'], random_state=SEED)
temp = df.set_index('row_id').loc[temp_ids]
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50,
                                     stratify=temp['stroke'], random_state=SEED)

df['split'] = 'train'
df.loc[df['row_id'].isin(val_ids),  'split'] = 'validation'
df.loc[df['row_id'].isin(test_ids), 'split'] = 'test'

summary = (df.groupby('split')
             .agg(respondents=('stroke', 'size'), stroke_cases=('stroke', 'sum'),
                  prevalence_pct=('stroke', lambda s: round(s.mean() * 100, 2)),
                  kept_primary=('meta_keep_primary', 'sum'))
             .loc[['train', 'validation', 'test']])
print(summary.to_string())
assert df['split'].value_counts().sum() == len(df)

### Step 8 — Save

In [ ]:
df.to_csv(READY_PATH, index=False)
df[['row_id', 'split']].to_csv(SPLIT_PATH, index=False)

config = {
    'seed': SEED,
    'core_missing_max': CORE_MISSING_MAX,
    'continuous': CONTINUOUS, 'ordinal': ORDINAL, 'nominal': NOMINAL, 'binary': BINARY,
    'features_A': FEATURES_A, 'features_B': FEATURES_B,
    'post_stroke_removed_in_B': POST_STROKE,
    'dropped': DROP,
    'target': 'stroke',
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print("=" * 60)
print("  Notebook 02 complete")
print("=" * 60)
print(f"  Respondents        : {len(df):,} (primary sample: {int(df['meta_keep_primary'].sum()):,})")
print(f"  Feature set A / B  : {len(FEATURES_A)} / {len(FEATURES_B)}")
print(f"  Saved data         : {READY_PATH}")
print(f"  Saved split        : {SPLIT_PATH}")
print(f"  Saved config       : {CONFIG_PATH}")
print(f"  Saved fill checks  : {CHECKS_PATH}")
print("=" * 60)